In [38]:
import wandb
import torchmetrics
import torch

import pytorch_lightning as pl

from torch.nn import functional as F

from torchvision.models import resnet18, resnet50, vgg

from torch import nn

from xaikd import datasets


# Introduction

Training Notebook: https://colab.research.google.com/drive/13NNSnXyRuN4vti22kKE2-vpQpTFGO0ta#scrollTo=IOD4IbN1tJhI&uniqifier=1

In [9]:
run = wandb.init()

In [4]:
class ModelWrapper(pl.LightningModule):
    def __init__(self, encoder):
        super().__init__()

        self.encoder = encoder

        num_classes = 100
        print(f"We have {num_classes} classes")


        self.metrics = dict(
            train=torchmetrics.Accuracy(task="multiclass", num_classes=num_classes),
            val=torchmetrics.Accuracy(task="multiclass", num_classes=num_classes)
        )

    def forward(self, x):
        embedding = self.encoder(x)
        return embedding

    def configure_optimizers(self):

        # ref: https://github.com/weiaicunzai/pytorch-cifar100/blob/master/train.py#L150
        # followed: https://arxiv.org/pdf/1708.04552.pdf
        optimizer = torch.optim.SGD(self.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
        scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[60, 120, 160], gamma=0.2)


        return [optimizer], [scheduler]


    def compute_metric(self, batch, prefix):

        x, y = batch

        logits = self.encoder(x)

        loss = F.cross_entropy(logits, y)

        self.log(f'{prefix}_loss', loss, on_epoch=True)
        self.metrics[prefix].update(logits.detach().cpu(), y.cpu())

        return loss

    def training_step(self, train_batch, batch_idx):

        loss = self.compute_metric(train_batch, 'train')

        return loss

    def summary_metric(self, prefix):

        metric = self.metrics[prefix]
        value = metric.compute()

        self.log(f'{prefix}_acc', value)

        metric.reset()


    def on_train_epoch_end(self):

        self.summary_metric('train')


    def validation_step(self, val_batch, batch_idx):
        x, y = val_batch

        logits = self.encoder(x)

        loss = self.compute_metric(val_batch, 'val')


    def on_validation_epoch_end(self):

        self.summary_metric('val')

In [5]:
NUM_CLASSES = 100

In [6]:
def create_resnet18():
    model = resnet18(num_classes=NUM_CLASSES)

    # ref: https://github.com/lightly-ai/lightly/blob/b69b8b14c29121422479f23078488efca734a995/lightly/models/resnet.py#L4
    model.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
    model.maxpool = nn.Identity()

    model.avgpool = nn.AvgPool2d(kernel_size=4)

    return model

def create_resnet50():
    model = resnet50(num_classes=NUM_CLASSES)

    # ref: https://github.com/lightly-ai/lightly/blob/b69b8b14c29121422479f23078488efca734a995/lightly/models/resnet.py#L4
    model.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
    model.maxpool = nn.Identity()
    model.avgpool = nn.AvgPool2d(kernel_size=4)

    return model

def create_vgg11():
    model = vgg.vgg11(num_classes=NUM_CLASSES)

    return model

In [7]:
def build_model(arch):
    if arch == "resnet18":
        return create_resnet18()
    elif arch == "resnet50":
        return create_resnet50()
    elif arch == "vgg11":
        return create_vgg11()

In [47]:
def load_run_id(arch, run_id, suffix):

    slug = f"p16i/xaikd-training-teacher-models/{run_id}"
    artifact = run.use_artifact(slug, type='model')
    artifact_dir = artifact.download()

    print(f"loading {run_id} (actual_name={artifact_dir})")
    model = ModelWrapper.load_from_checkpoint(
        f"{artifact_dir}/model.ckpt", map_location=torch.device('cpu'),
        encoder=build_model(arch)
    )
    model.eval()
    
    

    ds = datasets.construct("cifar100")
    
    ds_val = ds.create_subset(train_split=False)
    
    dl_val = datasets.build_dataloader(ds_val, shuffle=False)
    
    trainer = pl.Trainer()

    acc = trainer.validate(model, dataloaders=dl_val)
    
        
    print(f"saving {run_id}")
    output = f"./artifacts/converted/cifar100-{arch}-{suffix}--{run_id}.pth"
    
    print(f"to {output}")
    
    torch.save(model.encoder.state_dict(), output)

In [48]:
load_run_id("resnet18", "model-sszu9jtz:best", "v1")

wandb: Downloading large artifact model-sszu9jtz:best, 85.70MB. 1 files... 
wandb:   1 of 1 files downloaded.  
Done. 0:0:0.5
/home/pat/.cache/pypoetry/virtualenvs/xaikd-bYmevfGI-py3.11/lib/python3.11/site-packages/pytorch_lightning/utilities/migration/utils.py:55: PossibleUserWarning: The loaded checkpoint was produced with Lightning v2.0.9, which is newer than your current Lightning version: v2.0.7
  rank_zero_warn(


loading model-sszu9jtz:best (actual_name=./artifacts/model-sszu9jtz:v32)
We have 100 classes


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


Validation: 0it [00:00, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Runningstage.validating metric      DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
         val_acc            0.7763000130653381
        val_loss            0.9117957949638367
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
saving model-sszu9jtz:best
to ./artifacts/converted/cifar100-resnet18-v1--model-sszu9jtz:best.pth


wandb: 429 encountered (Filestream rate limit exceeded, retrying in 2.0 seconds.), retrying request
wandb: 429 encountered (Filestream rate limit exceeded, retrying in 4.3 seconds.), retrying request
wandb: 429 encountered (Filestream rate limit exceeded, retrying in 8.5 seconds.), retrying request
wandb: 429 encountered (Filestream rate limit exceeded, retrying in 2.4 seconds.), retrying request
wandb: 429 encountered (Filestream rate limit exceeded, retrying in 4.1 seconds.), retrying request
wandb: 429 encountered (Filestream rate limit exceeded, retrying in 8.4 seconds.), retrying request
wandb: 429 encountered (Filestream rate limit exceeded, retrying in 2.2 seconds.), retrying request
wandb: 429 encountered (Filestream rate limit exceeded, retrying in 4.3 seconds.), retrying request
wandb: 429 encountered (Filestream rate limit exceeded, retrying in 8.5 seconds.), retrying request


In [49]:
load_run_id("resnet18", "model-8no232l1:best", "v2")

wandb: Downloading large artifact model-8no232l1:best, 85.70MB. 1 files... 
wandb:   1 of 1 files downloaded.  
Done. 0:0:6.0


loading model-8no232l1:best (actual_name=./artifacts/model-8no232l1:v36)
We have 100 classes


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


Validation: 0it [00:00, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Runningstage.validating metric      DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
         val_acc            0.7824000120162964
        val_loss            0.8895390629768372
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
saving model-8no232l1:best
to ./artifacts/converted/cifar100-resnet18-v2--model-8no232l1:best.pth


In [50]:
load_run_id("resnet50", "model-dxngvotm:best", "v1")

wandb: Downloading large artifact model-dxngvotm:best, 181.21MB. 1 files... 
wandb:   1 of 1 files downloaded.  
Done. 0:0:6.7


loading model-dxngvotm:best (actual_name=./artifacts/model-dxngvotm:v28)
We have 100 classes


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


Validation: 0it [00:00, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Runningstage.validating metric      DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
         val_acc            0.7889999747276306
        val_loss            0.9427488446235657
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
saving model-dxngvotm:best
to ./artifacts/converted/cifar100-resnet50-v1--model-dxngvotm:best.pth


In [ ]:
#wip
# load_run_id("vgg11", "model-...", "v1")